In [1]:
import sys
import pandas as pd
import ace_lib as ace
import nest_asyncio
import asyncio
from openai import AsyncOpenAI

nest_asyncio.apply()
_llm_instance = None


async def call_llm(prompt):
    """
    Interface with the LLM API to process the given prompt.
    Consultants will modify this function to use their preferred LLM.
    """
    try:
        # Example: OpenAI GPT (consultants can replace this with their own LLM logic)
        client = AsyncOpenAI(
            base_url="...",
            # api_key = "your-api-key"
        )
        
        # Send the prompt to the chat completion endpoint
        response = await client.chat.completions.create(
            model="gpt-4",  # Specify the model
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error calling LLM: {e}")
        return None

# Generate English Description for Alpha
async def generate_alpha_description(alpha_id, brain_session):
    alpha_details = brain_session.get(f"https://api.worldquantbrain.com/alphas/{alpha_id}").json()
    alpha_expression = alpha_details['regular']['code']

    # If needed get operators or other data
    operators = ace.get_operators(brain_session)
    dataset_ids = ['pv1', 'shortinterest3']
    data_fields = pd.concat(
        [ace.get_datafields(brain_session, region='EUR', universe='TOP2500', dataset_id=dataset_id, data_type='ALL') for dataset_id in dataset_ids],
        ignore_index=True
    )

    # Generate English description using call_llm
    prompt = f"""Describe the following alpha in plain English:\n
    Alpha: {alpha_expression}. Here you can find used operators {operators[operators['scope']=='REGULAR'].to_json()}
    and data {data_fields.to_json()}."""
    description = await call_llm(prompt)
    return description.strip()

# Generate new Alphas based on generated description
async def generate_new_alphas(alpha_description, brain_session):
    num_alphas = 5

    # If needed get operators or other data
    operators = ace.get_operators(brain_session)
    dataset_ids = ['pv1', 'shortinterest3',]
    data_fields = pd.concat(
        [ace.get_datafields(brain_session, region='EUR', universe='TOP2500', dataset_id=dataset_id, data_type='ALL') for dataset_id in dataset_ids],
        ignore_index=True
    )

    prompt = f"""
    Based on the following description: '{alpha_description}', generate {num_alphas} new alpha expressions using the provided operators and data.

    Operators: {operators[operators['scope']=='REGULAR'].to_json()}, data {data_fields.to_json()} where id is data field name
    Important: You can use type=MATRIX field by itself, as input to Arithmetic, 
    Cross Sectional, Time Series operators, With Logical and Transformational operators, As group in Group operators, with bucket().
    You can’t use type=VECTOR field by itself. You only can use type=VECTOR field as input to Vector operator. Then you can treat it as a MATRIX field.
    Always wrap type=VECTOR data in category=Vector operator.
    You can’t use type=GROUP field by itself. You need to use it as “group” parameter in Group operator.

    Provide only {num_alphas} alpha expressions, they should not be the same.
    """
    response = await call_llm(prompt)
    return response.strip()


async def main():
    # Start Brain session
    brain_session = ace.start_session()

    # List your alpha IDs
    alpha_ids = ["..."]

    for alpha_id in alpha_ids:
        print(f"Processing Alpha ID: {alpha_id}")

        # Step 1: Generate English description of the alpha
        alpha_description = await generate_alpha_description(alpha_id, brain_session)
        print(f"\nAlpha Description:\n{alpha_description}")

        # Step 2: Generate new alphas based on the description
        new_alphas = await generate_new_alphas(alpha_description, brain_session)
        
        # Logic to simulate and tag
#         simulate_data = ace.generate_alpha(brain_session, regular=...)
#         simulation_result = ace.simulate_single_alpha(brain_session, simulate_data)
#         child_alpha_id = simulation_result['alpha_id']
#         ace.set_alpha_properties(brain_session, child_alpha_id, tags = [f"alpha_id"], regular_desc = generate_alpha_description(child_alpha_id, brain_session)) 
        
        print(f"\nNew Alphas:\n{new_alphas}")

In [2]:
asyncio.run(main())

Processing Alpha ID: 8x1X1wv

Alpha Description:
Here’s what the alpha is doing, step by step, in plain English:

- Measure each stock’s recent trading liquidity as 21-day average dollar volume: ts_mean(close × volume, 21).

- Put stocks into broad liquidity buckets by percentile (0–1, step 0.2) and “densify” them so only buckets that actually have stocks are kept. Then form a grouping gr that is the cartesian product of country and liquidity bucket. This lets the model treat a stock within its country and liquidity cohort.

- Build a short-interest “crowding” score:
  - Take the securities-lending “bar” rating (shrt3_bar, a 1–10 demand-to-borrow score), average it, and backfill up to 5 days to cover gaps.
  - Neutralize this score within the country × liquidity-bucket groups (to remove country and liquidity effects).
  - Scale the result. This yields rating: a crowding measure that’s comparable across countries and liquidity levels.

- Create the core alpha within industries:
  - Comp